## JTBD Recommendations - From Jobs to Products

Start from job to be done, articulate causal factors, connect to product/service group, to Walmart product or service.

Then, given Walmart product or service and job to be done, recommend other Walmart products or services that serve the JTBD and have complementary causal factors.

#### Prompts
0. System prompt to orient OpenAI models to perspective of JTBD researcher.
1. From JTBD to causal factors enabling progress on JTBD.
2. From causal factors to product or service categories.

#### Additional steps
3. Format prompt outputs for systematic collection.
4. From product or service category to Walmart product or service.
5. Create "JTBD recommendations": From individual Walmart product or service to other Walmart products or services. 
6. Compare current and JTBD recommendations.

#### Iteration ideas
- Ensemble results from multiple API calls using same prompt
- Extend to other JTBDs
- Link to outputs generated from reverse direction, from products to jobs

In [2]:
import openai
import os
import json
import csv
import pandas as pd
import pickle

openai.api_type = "azure"
openai.api_base = "https://[your azure openai base].openai.azure.com/"
openai.api_version = "2023-03-15-preview"
openai.api_key = "[your azure openai api key]"

### Initialize a conversation
A list of conversational turns taken by `system`, `user`, and `assistant`.

In [3]:
conversation = []

### Prompt 0: Prompt system to mimic JTBD researcher

In [4]:
system_prompt = """
You are a clinician doing jobs to be done (JTBD) research; you are familiar with JTBD product and user experience research techniques popularized by Jim Kalbach and Bob Moesta.
You make observations consistent with clinical evidence.
"""

In [5]:
conversation.append(
    {
        "role":"system",
        "content": system_prompt
        },
)

### Prompt 1: JTBD to Causal Factors

In [6]:
jtbd = "Minimize seasonal allergy symptoms"
prompt1 = f"""
For the JTBD of '{jtbd},' what are all of the causal factors that enable a customer to make progress against the JTBD?
Do not list types of products, medications, services, or behaviors in the response.
Only list the factors that these solutions deliver that cause progress.
Do not use AND or OR, and put each factor should be in a separate line.
Format the response as JSON, where the list of factors is named 'factors'.
"""
print(prompt1)


For the JTBD of 'Minimize seasonal allergy symptoms,' what are all of the causal factors that enable a customer to make progress against the JTBD?
Do not list types of products, medications, services, or behaviors in the response.
Only list the factors that these solutions deliver that cause progress.
Do not use AND or OR, and put each factor should be in a separate line.
Format the response as JSON, where the list of factors is named 'factors'.



In [7]:
conversation.append(
    {
        "role":"user",
        "content": prompt1
        },    
)

In [8]:
response = openai.ChatCompletion.create(
    engine="gpt-35-turbo", # azure api
    messages = conversation,
  temperature=0.7,
  max_tokens=3600,
  top_p=0.95,
  frequency_penalty=0,
  presence_penalty=0,
  stop=None)

# append response to conversation
conversation.append(
    {
        "role": "assistant", 
        "content": response['choices'][0]['message']['content']
        }
)

In [9]:
print(response.choices[0].message.content)

{
  "factors": [
    "Reduced exposure to allergens",
    "Increased resilience to allergens",
    "Improved immune system function",
    "Improved respiratory function",
    "Better sleep quality",
    "Improved mental health and well-being",
    "Improved physical comfort",
    "Improved cognitive function"
  ]
}


In [10]:
reso1 = json.loads(response.choices[0].message.content)

In [11]:
factors = reso1['factors']
reso1['factors']

['Reduced exposure to allergens',
 'Increased resilience to allergens',
 'Improved immune system function',
 'Improved respiratory function',
 'Better sleep quality',
 'Improved mental health and well-being',
 'Improved physical comfort',
 'Improved cognitive function']

#### Human in the loop opportunity: Review factors for credibility and compliance
May want to remove factors that reflect conventional wisdom / superstition rather than medical science, etc.

If factor is flagged for removal, queue for integration into system prompt.

In [12]:
df1 = pd.DataFrame({
    'jtbd': jtbd,
    'factor': factors
    })

In [13]:
df1

,jtbd,factor
0,Minimize seasonal allergy symptoms,Reduced exposure to allergens
1,Minimize seasonal allergy symptoms,Increased resilience to allergens
2,Minimize seasonal allergy symptoms,Improved immune system function
3,Minimize seasonal allergy symptoms,Improved respiratory function
4,Minimize seasonal allergy symptoms,Better sleep quality
5,Minimize seasonal allergy symptoms,Improved mental health and well-being
6,Minimize seasonal allergy symptoms,Improved physical comfort
7,Minimize seasonal allergy symptoms,Improved cognitive function


### Prompt 2: Causal Factors to Product and Service Categories

In [14]:
prompt2 = """
For each factor, provide a list of products and services, grouped by retail, medication, clinical service, or behavior.
Be comprehensive with the list.
Format the response as JSON. 
"""

# where 'factors' is a list of JSON objects, and each .

In [15]:
conversation.append(
    {
        "role":"user",
        "content": prompt2
        },    
)

In [16]:
response = openai.ChatCompletion.create(
    engine="gpt-35-turbo", # azure api
    messages = conversation,
  temperature=0.7,
  max_tokens=3600,
  top_p=0.95,
  frequency_penalty=0,
  presence_penalty=0,
  stop=None)

# append response to conversation
conversation.append(
    {
        "role": "assistant", 
        "content": response['choices'][0]['message']['content']
        }
)

In [17]:
print(response.choices[0].message.content)

{
  "factors": [
    {
      "factor": "Reduced exposure to allergens",
      "solutions": [
        {
          "retail": [
            "Air purifiers",
            "Dehumidifiers",
            "Vacuum cleaners with HEPA filters",
            "Allergy-friendly bedding",
            "Allergy-friendly cleaning products"
          ]
        },
        {
          "behavior": [
            "Keeping windows closed during high-pollen periods",
            "Taking shoes off before entering the house",
            "Showering before bedtime",
            "Wearing a mask while doing yard work"
          ]
        }
      ]
    },
    {
      "factor": "Increased resilience to allergens",
      "solutions": [
        {
          "medication": [
            "Allergy shots",
            "Sublingual allergy drops"
          ]
        },
        {
          "clinical service": [
            "Allergen immunotherapy",
            "Allergy testing"
          ]
        }
      ]
    },
    {
      "fact

In [18]:
reso2 = json.loads(response.choices[0].message.content)
grouped_solutions = reso2['factors']
grouped_solutions

[{'factor': 'Reduced exposure to allergens',
  'solutions': [{'retail': ['Air purifiers',
     'Dehumidifiers',
     'Vacuum cleaners with HEPA filters',
     'Allergy-friendly bedding',
     'Allergy-friendly cleaning products']},
   {'behavior': ['Keeping windows closed during high-pollen periods',
     'Taking shoes off before entering the house',
     'Showering before bedtime',
     'Wearing a mask while doing yard work']}]},
 {'factor': 'Increased resilience to allergens',
  'solutions': [{'medication': ['Allergy shots', 'Sublingual allergy drops']},
   {'clinical service': ['Allergen immunotherapy', 'Allergy testing']}]},
 {'factor': 'Improved immune system function',
  'solutions': [{'medication': ['Immunomodulators']},
   {'clinical service': ['IV vitamin therapy', 'Nutrition counseling']},
   {'behavior': ['Eating a balanced diet',
     'Getting regular exercise',
     'Reducing stress']}]},
 {'factor': 'Improved respiratory function',
  'solutions': [{'medication': ['Broncho

In [26]:
df2 = pd.DataFrame()

for factorJSON in grouped_solutions:
    factor = factorJSON['factor']
    print()
    print(factor)
    for solutionJSON in factorJSON['solutions']:
        for (solution_category, solution_items) in solutionJSON.items():
            print(solution_category)
            dfTemp = pd.DataFrame(
                {
                'factor': factor,
                'solution': solution_category,
                'prod_svc_type': solution_items,
                }
            )
            df2 = pd.concat([df2, dfTemp])


Reduced exposure to allergens
retail
behavior

Increased resilience to allergens
medication
clinical service

Improved immune system function
medication
clinical service
behavior

Improved respiratory function
medication
clinical service
behavior

Better sleep quality
retail
medication
clinical service
behavior

Improved mental health and well-being
medication
clinical service
behavior

Improved physical comfort
retail
medication
clinical service
behavior

Improved cognitive function
medication
clinical service
behavior


In [27]:
df2.head(20)

,factor,solution,prod_svc_type
0,Reduced exposure to allergens,retail,Air purifiers
1,Reduced exposure to allergens,retail,Dehumidifiers
2,Reduced exposure to allergens,retail,Vacuum cleaners with HEPA filters
3,Reduced exposure to allergens,retail,Allergy-friendly bedding
4,Reduced exposure to allergens,retail,Allergy-friendly cleaning products
0,Reduced exposure to allergens,behavior,Keeping windows closed during high-pollen periods
1,Reduced exposure to allergens,behavior,Taking shoes off before entering the house
2,Reduced exposure to allergens,behavior,Showering before bedtime
3,Reduced exposure to allergens,behavior,Wearing a mask while doing yard work
0,Increased resilience to allergens,medication,Allergy shots


In [28]:
# remove behaviors (for now)
df3  = df2[df2.solution != "behavior"].copy()
df3.head(20)

,factor,solution,prod_svc_type
0,Reduced exposure to allergens,retail,Air purifiers
1,Reduced exposure to allergens,retail,Dehumidifiers
2,Reduced exposure to allergens,retail,Vacuum cleaners with HEPA filters
3,Reduced exposure to allergens,retail,Allergy-friendly bedding
4,Reduced exposure to allergens,retail,Allergy-friendly cleaning products
0,Increased resilience to allergens,medication,Allergy shots
1,Increased resilience to allergens,medication,Sublingual allergy drops
0,Increased resilience to allergens,clinical service,Allergen immunotherapy
1,Increased resilience to allergens,clinical service,Allergy testing
0,Improved immune system function,medication,Immunomodulators


#### Next steps 
Next layer detail would be helpful from prompt: Which types of air purifiers?
Don't want all air purifiers. 

Want a ranking relating the product/service to the particular causal factor.

### Step 3. Engineer Prompts 1 and 2 to reliably generate JSON

Need outputs to be well structured to enable easier loading into tables and databases.

### Step 4. From OpenAI product or service category to Walmart product or service.
This step depends on a way of searching or linking a product or service category or type generated by OpenAI model. 

That is, we will have natural language outputs like "Corticosteroids (e.g. prednisone, fluticasone)" that need to be connected to all the Walmart products in that category.

### Step 5. Create "JTBD recommendations"
From one Walmart product or service to others that enable progress towards the JTBD by means of other causal factors.

Current state Walmart.com recommendations, presented as "More items to consider," are "based on what others bought," using basket analysis from prior shopping behavior. See ![nyquil basket-based recommendations](img.png)

JTBD recommendations of "More items to consider" would be based on the customer's presumed or predicted job, and a hypothesis that they may be looking for additional items that help the customer make progress on the job to be done. 

Products and services can have overlapping or dissimilar causal factors that affect their ability to help the customer with the JTBD. 

#### Idea 5.1: Pre-purchase JTBD recommendations
If two items have completely overlapping or very similar sets of causal factors, they will be substitutes for each other wrt the JTBD. If the customer hasn't purchased an item with those causal factors, they may value seeing items that share causal factors.

#### Idea 5.2: Post-purchase JTBD recommendations
It two items have completely distinct or dissimilar causal factors, they are complements wrt the JTBD. Once the customer has purchased an item, the customer may value seeing items that offer complementary causal factors for progressing wrt the JTBD.


Until Walmart team completes Step 4, we can explore recommendations at the category level instead of the lower-level, named product/service level.

#### Calculate similarity in causal factors between pairs of products or services

Initially, we'll use product/service categories rather than products directly.

In [29]:
df3.factor.value_counts()

Improved physical comfort                11
Better sleep quality                      6
Reduced exposure to allergens             5
Increased resilience to allergens         4
Improved respiratory function             4
Improved mental health and well-being     4
Improved immune system function           3
Improved cognitive function               3
Name: factor, dtype: int64

In [30]:
prod_svc_factors = df3.groupby('prod_svc_type', as_index=False)['factor'].agg(list)

In [31]:
prod_svc_factors

,prod_svc_type,factor
0,Acupuncture,[Improved physical comfort]
1,Air purifiers,[Reduced exposure to allergens]
2,Allergen immunotherapy,[Increased resilience to allergens]
3,Allergy shots,[Increased resilience to allergens]
4,Allergy testing,[Increased resilience to allergens]
5,Allergy-friendly bedding,"[Reduced exposure to allergens, Better sleep q..."
6,Allergy-friendly cleaning products,[Reduced exposure to allergens]
7,Antidepressants,[Improved mental health and well-being]
8,Antihistamines,[Improved physical comfort]
9,Anxiolytics,[Improved mental health and well-being]


#### Create Jaccard similarity matrix

In [36]:
ddf = pd.get_dummies(prod_svc_factors.factor.explode()).groupby(level=0).sum().set_index(prod_svc_factors.prod_svc_type)

In [37]:
ddf

,Better sleep quality,Improved cognitive function,Improved immune system function,Improved mental health and well-being,Improved physical comfort,Improved respiratory function,Increased resilience to allergens,Reduced exposure to allergens
prod_svc_type,,,,,,,,
Acupuncture,0,0,0,0,1,0,0,0
Air purifiers,0,0,0,0,0,0,0,1
Allergen immunotherapy,0,0,0,0,0,0,1,0
Allergy shots,0,0,0,0,0,0,1,0
Allergy testing,0,0,0,0,0,0,1,0
Allergy-friendly bedding,1,0,0,0,0,0,0,1
Allergy-friendly cleaning products,0,0,0,0,0,0,0,1
Antidepressants,0,0,0,1,0,0,0,0
Antihistamines,0,0,0,0,1,0,0,0


In [82]:
from sklearn.metrics.pairwise import pairwise_distances
import numpy as np

jnp = 1 - pairwise_distances(ddf.to_numpy(), metric='jaccard')
np.fill_diagonal(jnp, np.nan)
jdf = pd.DataFrame(
    jnp, 
    columns=list(range(len(prod_svc_factors.prod_svc_type.values))),
    index=list(range(len(prod_svc_factors.prod_svc_type.values))),
    )

/anaconda/envs/azureml_py310_sdkv2/lib/python3.10/site-packages/sklearn/metrics/pairwise.py:2025: DataConversionWarning: Data was converted to boolean for metric jaccard
  warnings.warn(msg, DataConversionWarning)


In [83]:
jdf

,0,1,2,3,4,5,6,7,8,9,...,29,30,31,32,33,34,35,36,37,38
0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
1,0.0,NaN,0.0,0.0,0.0,0.5,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2,0.0,0.0,NaN,1.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
3,0.0,0.0,1.0,NaN,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
4,0.0,0.0,1.0,1.0,NaN,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
5,0.0,0.5,0.0,0.0,0.0,NaN,0.5,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.5,0.5,0.0,0.5,0.5
6,0.0,1.0,0.0,0.0,0.0,0.5,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
7,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,1.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,NaN,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


#### Convert to edgelist

In [84]:
edges = jdf.stack().reset_index().rename(columns= {'level_0':'From', 'level_1':'To', 0:'Jaccard_similarity'})
edges.head(10)

,From,To,Jaccard_similarity
0,0,1,0.0
1,0,2,0.0
2,0,3,0.0
3,0,4,0.0
4,0,5,0.0
5,0,6,0.0
6,0,7,0.0
7,0,8,1.0
8,0,9,0.0
9,0,10,0.0


In [85]:
import networkx as nx

G = nx.from_pandas_edgelist(
    edges[edges['Jaccard_similarity']>0],
    source='From',
    target='To',
    edge_attr='Jaccard_similarity',
)

In [104]:
G.edges(data=True)

EdgeDataView([(0, 8, {'Jaccard_similarity': 1.0}), (0, 12, {'Jaccard_similarity': 1.0}), (0, 17, {'Jaccard_similarity': 1.0}), (0, 18, {'Jaccard_similarity': 1.0}), (0, 23, {'Jaccard_similarity': 1.0}), (0, 25, {'Jaccard_similarity': 1.0}), (0, 26, {'Jaccard_similarity': 1.0}), (0, 27, {'Jaccard_similarity': 1.0}), (0, 30, {'Jaccard_similarity': 1.0}), (0, 33, {'Jaccard_similarity': 1.0}), (8, 12, {'Jaccard_similarity': 1.0}), (8, 17, {'Jaccard_similarity': 1.0}), (8, 18, {'Jaccard_similarity': 1.0}), (8, 23, {'Jaccard_similarity': 1.0}), (8, 25, {'Jaccard_similarity': 1.0}), (8, 26, {'Jaccard_similarity': 1.0}), (8, 27, {'Jaccard_similarity': 1.0}), (8, 30, {'Jaccard_similarity': 1.0}), (8, 33, {'Jaccard_similarity': 1.0}), (12, 17, {'Jaccard_similarity': 1.0}), (12, 18, {'Jaccard_similarity': 1.0}), (12, 23, {'Jaccard_similarity': 1.0}), (12, 25, {'Jaccard_similarity': 1.0}), (12, 26, {'Jaccard_similarity': 1.0}), (12, 27, {'Jaccard_similarity': 1.0}), (12, 30, {'Jaccard_similarity':

In [71]:
# from pyvis.network import Network
# net = Network()#notebook=True)
# net.from_nx(G)
# net.show(f'{jtbd} - causal factor similarity edges.html', local=False, notebook=False)

Minimize seasonal allergy symptoms - causal factor similarity edges.html


In [114]:
from bokeh.palettes import magma
from bokeh.plotting import figure, from_networkx, show
from bokeh.io import output_notebook
from bokeh.models import (
    BoxZoomTool, ResetTool,
    # BoxSelectTool, NodesAndLinkedEdges, LassoSelectTool, WheelZoomTool, 
    )

output_notebook()

Loading BokehJS ...

In [115]:
p = figure(x_range=(-2, 2), y_range=(-2, 2),
           x_axis_location=None, y_axis_location=None,
           tools="hover", tooltips="@prod_svc_type"
        )
p.grid.grid_line_color = None
p.add_tools(
        BoxZoomTool(), 
        ResetTool()
        )

graph = from_networkx(G, nx.spring_layout, scale=1.9, center=(0,0))
p.renderers.append(graph)

# Add metadata to nodes 
graph.node_renderer.data_source.data['index'] = list(G.nodes)
graph.node_renderer.data_source.data['colors'] = magma(len(G.nodes))
graph.node_renderer.data_source.data['prod_svc_type'] = [prod_svc_factors.prod_svc_type.values[i] for i in list(G.nodes)]

graph.node_renderer.glyph.update(size=12, fill_color="colors")


show(p)

[bokeh for network graphs](https://docs.bokeh.org/en/latest/docs/user_guide/topics/graph.html)

### Step 6. Compare current and JTBD recommendations.
